# VAE x64 Beta Sweep

This notebook is the curated submission-facing entry point for the VAE `beta` sweep on the shared `x64` anomaly benchmark.
It defaults to reusing the saved per-beta artifacts instead of rerunning the sweep.


## Submission Context

- Dataset notebook: `data/dataset/x64/benchmark_50k_5pct/notebook.ipynb`
- Sweep config: `experiments/anomaly_detection/vae/x64/beta_sweep/train_config.toml`
- Training script: `scripts/train_vae.py`
- Artifact root: `experiments/anomaly_detection/vae/x64/beta_sweep/artifacts/vae_beta_sweep/`
- Default behavior: reuse the saved `results/beta_sweep_summary.json` and the per-beta checkpoints and evaluation outputs


### Setup And Imports

This cell resolves the repo root, exposes `src/` on `PYTHONPATH`, and imports the shared config loader plus the standard utilities used to read the saved sweep artifacts.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj: object) -> None:
        print(obj)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'wafer_defect').exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / 'src' / 'wafer_defect').exists():
            REPO_ROOT = candidate
            break

SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml


### Load The Sweep Config And Run Controls

This cell points the notebook at the local beta-sweep config and exposes the rerun flags. Leave them as `False` to keep the notebook in artifact-first mode.


In [ ]:
CONFIG_PATH = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'vae' / 'x64' / 'beta_sweep' / 'train_config.toml'
config = load_toml(CONFIG_PATH)

SWEEP_ROOT = REPO_ROOT / config['run']['output_dir']
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)
CONFIGS_DIR = SWEEP_ROOT / 'configs'
CONFIGS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = SWEEP_ROOT / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RERUN_SWEEP = False
RUN_BETA_SWEEP = False

display(config)


### Optionally Rerun The Sweep

This cell is only used when you explicitly enable the sweep rerun. Otherwise it simply reuses the saved summary and per-beta artifacts already present in the repository.


In [ ]:

def format_toml_value(value):
    if isinstance(value, bool):
        return 'true' if value else 'false'
    if isinstance(value, (int, float)):
        return repr(value)
    escaped = str(value).replace('\\', '\\\\').replace('"', '\\"')
    return f'"{escaped}"'

def dump_toml(config_dict):
    lines = []
    for section, values in config_dict.items():
        lines.append(f'[{section}]')
        for key, value in values.items():
            lines.append(f'{key} = {format_toml_value(value)}')
        lines.append('')
    return '\n'.join(lines).rstrip() + '\n'

def beta_tag(beta_value: float) -> str:
    return str(beta_value).replace('.', 'p')

summary_path = SWEEP_ROOT / 'results' / 'beta_sweep_summary.json'
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC_ROOT) if not env.get('PYTHONPATH') else str(SRC_ROOT) + os.pathsep + env['PYTHONPATH']

if RUN_BETA_SWEEP or FORCE_RERUN_SWEEP or not summary_path.exists():
    beta_values = list(config['model']['beta_values'])
    results = []
    for beta_value in beta_values:
        tag = beta_tag(float(beta_value))
        run_output_dir = SWEEP_ROOT / f'beta_{tag}'
        run_config = {
            'run': {'output_dir': run_output_dir.relative_to(REPO_ROOT).as_posix(), 'seed': config['run']['seed']},
            'data': dict(config['data']),
            'training': dict(config['training']),
            'model': {'type': 'vae', 'latent_dim': 128, 'beta': float(beta_value)},
        }
        run_config_path = CONFIGS_DIR / f'train_vae_beta_{tag}.toml'
        run_config_path.write_text(dump_toml(run_config), encoding='utf-8')

        train_cmd = [sys.executable, 'scripts/train_vae.py', '--config', str(run_config_path.relative_to(REPO_ROOT))]
        eval_cmd = [
            sys.executable,
            'scripts/evaluate_reconstruction_model.py',
            '--checkpoint', str((run_output_dir / 'checkpoints' / 'best_model.pt').relative_to(REPO_ROOT)),
            '--config', str(run_config_path.relative_to(REPO_ROOT)),
            '--output-dir', str((run_output_dir / 'evaluation').relative_to(REPO_ROOT)),
        ]
        subprocess.run(train_cmd, cwd=REPO_ROOT, env=env, check=True)
        subprocess.run(eval_cmd, cwd=REPO_ROOT, env=env, check=True)

        run_summary = json.loads((run_output_dir / 'results' / 'evaluation' / 'summary.json').read_text(encoding='utf-8'))
        results.append({
            'beta': float(beta_value),
            'threshold': float(run_summary['threshold']),
            'val_threshold_precision': float(run_summary['metrics_at_validation_threshold']['precision']),
            'val_threshold_recall': float(run_summary['metrics_at_validation_threshold']['recall']),
            'val_threshold_f1': float(run_summary['metrics_at_validation_threshold']['f1']),
            'auroc': float(run_summary['metrics_at_validation_threshold']['auroc']),
            'auprc': float(run_summary['metrics_at_validation_threshold']['auprc']),
            'best_sweep_threshold': float(run_summary['best_threshold_sweep']['threshold']),
            'best_sweep_precision': float(run_summary['best_threshold_sweep']['precision']),
            'best_sweep_recall': float(run_summary['best_threshold_sweep']['recall']),
            'best_sweep_f1': float(run_summary['best_threshold_sweep']['f1']),
        })

    summary_path.write_text(json.dumps({'betas': beta_values, 'results': results}, indent=2), encoding='utf-8')
else:
    print(f'Found existing beta sweep artifacts in {SWEEP_ROOT}. Skipping rerun.')


### Load The Saved Sweep Summary

This cell reads the aggregated summary and builds a dataframe that we can use for the comparison plots and tables.


In [ ]:
summary_payload = json.loads((SWEEP_ROOT / 'results' / 'beta_sweep_summary.json').read_text(encoding='utf-8'))
beta_sweep_df = pd.DataFrame(summary_payload['results']).sort_values('val_threshold_f1', ascending=False).reset_index(drop=True)
display(beta_sweep_df)


### Plot Sweep Metrics Across `beta`

This cell recreates the main sweep comparison figure, shows it inline, and saves it to the local `plots/` folder.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
sorted_df = beta_sweep_df.sort_values('beta').reset_index(drop=True)

axes[0].plot(sorted_df['beta'], sorted_df['val_threshold_f1'], marker='o', label='val-threshold F1')
axes[0].plot(sorted_df['beta'], sorted_df['best_sweep_f1'], marker='s', label='best-sweep F1')
axes[0].set_title('F1 vs Beta')
axes[0].set_xlabel('beta')
axes[0].legend()

axes[1].plot(sorted_df['beta'], sorted_df['auroc'], marker='o', color='#1f77b4')
axes[1].set_title('AUROC vs Beta')
axes[1].set_xlabel('beta')

axes[2].plot(sorted_df['beta'], sorted_df['auprc'], marker='o', color='#2ca02c')
axes[2].set_title('AUPRC vs Beta')
axes[2].set_xlabel('beta')

fig.savefig(PLOTS_DIR / 'beta_sweep_metrics.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)


### Compare Training Curves Across Betas

This cell loads each saved `history.json`, overlays the validation curves, and saves the comparison plot for the report.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
for beta_value in sorted(beta_sweep_df['beta'].tolist()):
    tag = str(beta_value).replace('.', 'p')
    history_path = SWEEP_ROOT / f'beta_{tag}' / 'results' / 'history.json'
    history_df = pd.DataFrame(json.loads(history_path.read_text(encoding='utf-8')))
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label=f'beta={beta_value}')
    axes[1].plot(history_df['epoch'], history_df['val_reconstruction_loss'], label=f'beta={beta_value}')

axes[0].set_title('Validation Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend(fontsize=8)
axes[1].set_title('Validation Reconstruction Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8)

fig.savefig(PLOTS_DIR / 'beta_sweep_training_curves.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)


### Inspect The Best Beta Run

This cell identifies the best saved beta run by validation-threshold F1 and displays its saved evaluation summary so the sweep still ends with a concrete recommended configuration.


In [ ]:
best_beta_row = beta_sweep_df.iloc[0].to_dict()
best_beta = float(best_beta_row['beta'])
best_tag = str(best_beta).replace('.', 'p')
best_run_dir = SWEEP_ROOT / f'beta_{best_tag}'
best_eval_summary = json.loads((best_run_dir / 'results' / 'evaluation' / 'summary.json').read_text(encoding='utf-8'))
best_test_scores_df = pd.read_csv(best_run_dir / 'results' / 'evaluation' / 'test_scores.csv')
best_threshold_sweep_df = pd.read_csv(best_run_dir / 'results' / 'evaluation' / 'threshold_sweep.csv')
cm = best_eval_summary['metrics_at_validation_threshold'].get('confusion_matrix', [[0, 0], [0, 0]])
cm_df = pd.DataFrame(cm, index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])

print(f'Best beta by validation-threshold F1: {best_beta}')
print(f'Best run directory: {best_run_dir}')
display(pd.DataFrame([best_beta_row]))
display(pd.DataFrame([best_eval_summary['metrics_at_validation_threshold']]))
display(pd.DataFrame([best_eval_summary['best_threshold_sweep']]))

default_threshold = float(best_eval_summary['threshold'])
best_sweep_threshold = float(best_eval_summary['best_threshold_sweep']['threshold'])
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), constrained_layout=True)
axes[0].hist(best_test_scores_df.loc[best_test_scores_df['is_anomaly'] == 0, 'score'], bins=30, alpha=0.7, label='normal')
axes[0].hist(best_test_scores_df.loc[best_test_scores_df['is_anomaly'] == 1, 'score'], bins=30, alpha=0.7, label='anomaly')
axes[0].axvline(default_threshold, color='red', linestyle='--', label=f'val threshold = {default_threshold:.4f}')
axes[0].set_title(f'Best-Beta Score Distribution (beta={best_beta})')
axes[0].set_xlabel('Anomaly score')
axes[0].legend()

axes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['precision'], label='precision')
axes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['recall'], label='recall')
axes[1].plot(best_threshold_sweep_df['threshold'], best_threshold_sweep_df['f1'], label='f1')
axes[1].axvline(default_threshold, color='red', linestyle='--', label='val threshold')
axes[1].axvline(best_sweep_threshold, color='green', linestyle=':', label='best sweep')
axes[1].set_title('Best-Beta Threshold Sweep')
axes[1].set_xlabel('Threshold')
axes[1].legend()

heatmap = axes[2].imshow(cm_df.to_numpy(), cmap='Blues')
axes[2].set_xticks(range(cm_df.shape[1]), cm_df.columns)
axes[2].set_yticks(range(cm_df.shape[0]), cm_df.index)
axes[2].set_title('Best-Beta Confusion Matrix')
axes[2].set_xlabel('Predicted label')
axes[2].set_ylabel('True label')
for row_idx in range(cm_df.shape[0]):
    for col_idx in range(cm_df.shape[1]):
        axes[2].text(col_idx, row_idx, str(int(cm_df.iat[row_idx, col_idx])), ha='center', va='center', color='black')
fig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)
fig.savefig(PLOTS_DIR / 'best_beta_distribution_sweep_confusion.png', dpi=160, bbox_inches='tight')
display(fig)
plt.close(fig)
display(cm_df)
